In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"


config = Config()

In [4]:
# %%time

# X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config)
# # X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
# #     config,
# #     X_train,
# #     X_val,
# #     X_val_seq,
# #     ordinal_encoder,
# #     return_only_val=True,
# #     truncate_val=True
# # )
# pad = ordinal_encoder.unknown_value

In [5]:
mod = "test"

In [6]:
%%time

dict_models_preds = dict()

cols_to_drop = ['index']

cols_to_rename = dict()
cols_to_rename[config.col_user] = 'session'
cols_to_rename['item_id'] = 'preds'

cols_to_add_model_name = ['preds']

model_names = ['rnn', 'weak_nns', #'dssm_ep1', #'popular_at_day', 
             #  'rnn_triplet', 
               'cooc_smart', 'bm25', 'tfidf', 'cosine', 'last_items']

for model_name in model_names:
    dict_models_preds[model_name] = pl.read_parquet(config.cur_model_output_path + f'cur_{model_name}_df_{mod}_preds_100.parquet')
    for col in cols_to_drop:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].drop(col)
    for col in cols_to_rename:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: cols_to_rename[col]})
    for col in cols_to_add_model_name:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: model_name + '_' + col})
    if 'rnn' in model_name:
        dict_models_preds[model_name] = dict_models_preds[model_name].rename({'preds_scores': f'{model_name}_score'})
        dict_models_preds[model_name] = dict_models_preds[model_name].with_column(
            pl.col(f'{model_name}_score')
                .rank(method="ordinal", reverse=True)
                .over(["session"])
                .alias(f'{model_name}_rank')
        )
        # if model_name in ['rnn_triplet']:
        #     dict_models_preds[model_name] = dict_models_preds[model_name].filter(pl.col(f'{model_name}_rank') <= 50)
    elif model_name in ['cooc_smart']:
        for col in ['cooc_triplet_score_0', 'cooc_triplet_rank_0', 'cooc_score_0', 'cooc_rank_0']:
            if col in dict_models_preds[model_name]:
                dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: col[:-2]})
    for col in dict_models_preds[model_name].columns:
        if 'preds' in col:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: 'item_id'})

for model_name in model_names:
    dict_models_preds[model_name] = dict_models_preds[model_name].with_columns(
    [
        pl.col("item_id").cast(pl.Int32, strict=False).alias("item_id"),
        pl.col("session").cast(pl.Int32, strict=False).alias("session"),
    ]
)

CPU times: user 2min 46s, sys: 23.7 s, total: 3min 10s
Wall time: 50.4 s


In [7]:
dict_models_preds["bm25"].head()

item_id,bm25_score,bm25_rank,session
i32,f64,i64,i32
1302440,9302.814338,1,13755520
1491840,3947.518871,2,13755520
1246868,1896.91528,3,13755520
1481497,1764.942077,4,13755520
167883,1663.869831,5,13755520


In [8]:
dict_models_preds["last_items"].head()

item_id,last_items_rank,session
i32,i64,i32
566383,1,14308864
582364,1,14041920
1776244,1,14450304
1681793,1,13307264
1258572,1,13611584


In [9]:
model_names

['rnn', 'weak_nns', 'cooc_smart', 'bm25', 'tfidf', 'cosine', 'last_items']

In [12]:
df_users

session
i64
12899779
12899780
12899781
12899782
12899783
12899784
12899785
12899786
12899787


In [13]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")

df_users = pl.from_pandas(df_test.to_pandas()[[config.col_user]].drop_duplicates())
df_users = df_users.rename({config.col_user: "session"})

df_users = df_users.with_columns(
    [
        pl.col("session").cast(pl.Int32, strict=False).alias("session"),
    ]
)

del df_test
gc.collect()

chunk_size = 100_000
chunks = [
    (i, i * chunk_size, ((i + 1) * chunk_size))
    for i in range(len(df_users) // chunk_size + 1)
]

for chunk_id, chunk_start, chunk_end in tqdm(chunks, desc='creating chunks'):
    chunk_name = f'{config.cur_model_output_path}test_{chunk_id}.parquet'
    chunk_ids = df_users.to_pandas().loc[chunk_start : chunk_end - 1]['session'].tolist()
    df_merged = df_users.filter(pl.col('session').is_in(chunk_ids))
    for i in tqdm(range(0, len(model_names)), desc='merging'):
        model_name_cur = model_names[i]
        df_cur = utils.cast_to_float32(dict_models_preds[model_name_cur]).filter(pl.col('session').is_in(chunk_ids))
        df_merged = utils.polars_outer_merge(df_merged, df_cur)
        df_merged = utils.cast_to_float32(df_merged)
    df_merged = df_merged.with_columns(
    [
        pl.col("item_id").cast(pl.Int32, strict=False).alias("item_id"),
        pl.col("session").cast(pl.Int32, strict=False).alias("session"),
    ]
)
    df_merged.filter(pl.col('session').is_in(chunk_ids)).write_parquet(chunk_name)

creating chunks:   0%|          | 0/17 [00:00<?, ?it/s]

merging:   0%|          | 0/7 [00:00<?, ?it/s]

merging:   0%|          | 0/7 [00:00<?, ?it/s]

merging:   0%|          | 0/7 [00:00<?, ?it/s]

merging:   0%|          | 0/7 [00:00<?, ?it/s]

merging:   0%|          | 0/7 [00:00<?, ?it/s]

... (9 outputs left out)

merging:   0%|          | 0/7 [00:00<?, ?it/s]

merging:   0%|          | 0/7 [00:00<?, ?it/s]

merging:   0%|          | 0/7 [00:00<?, ?it/s]

CPU times: user 35min 28s, sys: 7min 32s, total: 43min 1s
Wall time: 19min 47s


In [11]:
stop

NameError: name 'stop' is not defined

In [ ]:
%%time

df_merged = dict_models_preds[model_names[0]]
for i in tqdm(range(1, len(model_names))):
    model_name_cur = model_names[i]
    dict_models_preds[model_name_cur] = utils.cast_to_float32(dict_models_preds[model_name_cur])
    df_merged = utils.polars_outer_merge(df_merged, dict_models_preds[model_name_cur])
    df_merged = utils.cast_to_float32(df_merged)

    del dict_models_preds[model_name_cur]
    gc.collect()

    if i == 1:
        del dict_models_preds[model_names[0]]
        gc.collect()

del dict_models_preds
gc.collect()

In [ ]:
df_merged.head()

In [ ]:
%%time

df_merged.write_parquet(config.cur_model_output_path + 'test.parquet')

In [ ]:
PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")

In [ ]:
%%time

df_users = pl.from_pandas(df_test.to_pandas()[[config.col_user]].drop_duplicates())
df_users = df_users.rename({config.col_user: "session"})

In [ ]:
del df_test
gc.collect()

In [ ]:
df_users.to_pandas().loc[1671800:1671805]

In [ ]:
chunk_size = 250_000
chunks = [
    (i, i * chunk_size, ((i + 1) * chunk_size))
    for i in range(len(df_users) // chunk_size + 1)
]

In [ ]:
chunks

In [ ]:
%%time

for chunk_id, chunk_start, chunk_end in tqdm(chunks):
    chunk_name = f'{config.cur_model_output_path}test_{chunk_id}.parquet'
    chunk_ids = df_users.to_pandas().loc[chunk_start : chunk_end - 1]['session'].tolist()
    df_merged.filter(pl.col('session').is_in(chunk_ids)).write_parquet(chunk_name)

In [ ]:
stop

In [ ]:
%%time

df_merged = df_users.join(df_merged, how='left', on='session')

In [ ]:
stop

In [ ]:
# %%time

# X_val_lists = X_val.groupby([config.col_user]).agg(
#     [
#         pl.col(config.col_aid).list().alias(config.col_train_seq),
#     ]
# )

In [ ]:
# %%time

# df_targets = utils.create_targets_df(list(X_val_lists.to_pandas()[config.col_train_seq]))

In [ ]:
%%time

df_users = pl.from_pandas(X_val_lists.to_pandas()[[config.col_user]].reset_index())
df_users = df_users.join(df_targets, on='index', how='inner')
df_users = df_users.drop('index')
df_users = df_users.rename({config.col_user: 'session'})

In [ ]:
%%time

df_merged = utils.polars_outer_merge(df_merged, df_users)

In [ ]:
del df_users, dict_models_preds
gc.collect()

In [ ]:
columns_from_models = [
    col for col in df_merged.columns if "_score" in col or "_rank" in col
]

In [ ]:
# %%time

# df_merged = pl.from_pandas(df_merged.to_pandas().dropna(subset=columns_from_models, thresh=1))

In [ ]:
%%time

df_merged = df_merged.with_column(pl.col("target").fill_null(0))
df_merged = df_merged.with_columns(
        [
            pl.col('target')
            .cast(pl.Float32)
            .alias('target'),
        ]
    )

In [ ]:
df_merged.head()

In [ ]:
%%time

df_merged.write_parquet(config.cur_model_output_path + 'train.parquet')

In [ ]:
stop

In [ ]:
qids_train = df_merged.to_pandas().groupby(["session"])["item_id"].count().values

In [ ]:
qids_train

In [ ]:
from lightgbm import LGBMRanker

In [ ]:
df_merged

In [ ]:
def find_query(q: str, data: list) -> list:
    path_list = list()

    def helper(subdata, path):
        for index, item in enumerate(subdata):
            if isinstance(item, list):
                helper(item, path + [index])
            elif item == q:
                path_list.append(path + [index])

    helper(data, list())
    return path_list

In [ ]:
data = ["a", ["b", "c", ["d", "a"]]]

In [ ]:
find_query("d", data)

In [ ]:
import lightgbm as lgb

In [ ]:
bst = lgb.Booster(model_file="../src/otto_recommender/booster_fold2.lgb")

In [ ]:
importance_df = (
    pd.DataFrame(
        {
            "feature_name": bst.feature_name(),
            "importance_gain": bst.feature_importance(importance_type="gain"),
            "importance_split": bst.feature_importance(importance_type="split"),
        }
    )
    .sort_values("importance_gain", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
importance_df

In [ ]:
lgb.plot_importance(bst, importance_type="gain")

In [ ]:
df_merged = pl.read_parquet(config.cur_model_output_path + "train.parquet").to_pandas()

In [ ]:
%%time

columns = [
    col for col in df_merged.columns if "_score" in col or "_rank" in col
]
X_train = df_merged[columns].to_numpy(dtype=np.float32)
df_merged['target'] = (df_merged['target'] > 0) * 1
y_train = df_merged['target'].astype(np.int8)

In [ ]:
%%time

train_groups = utils.get_group_for_lgb(df_merged["session"].values)

In [ ]:
%%time

lgb_train = lgb.Dataset(
    X_train,
    y_train,
    feature_name=columns,
    group=train_groups,
    free_raw_data=False,
)

In [ ]:
%%time

lgb_train.construct()

In [ ]:
lgb_train.get_label()

In [ ]:
lgb_train.get_feature_name()

In [ ]:
len(lgb_train.get_data())

In [ ]:
%%time

len(np.where((np.isnan(lgb_train.get_data()).all(axis=1) == False) & (lgb_train.get_label() == 1))[0])

In [ ]:
df_merged.loc[
    np.where(
        (np.isnan(lgb_train.get_data()).all(axis=1) == False)
        & (lgb_train.get_label() == 1)
    )[0]
]

In [ ]:
946438 / len(df_merged[df_merged["target"] == 1])

In [ ]:
df_merged.loc[
    np.where(
        (np.isnan(lgb_train.get_data()).all(axis=1) == True)
        & (lgb_train.get_label() == 0)
    )[0]
]